In [ ]:
import torch

import psutil
import threading
import os

import copy
import numpy as np
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from matplotlib import rc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, MaxAbsScaler, RobustScaler

from torch import nn, optim, backends
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

import sys
sys.path.insert(0,'/ascldap/users/amtumli/dev/cert-rnn-py/src')
import cert_rnn
from cert_rnn import LSTMAutoencoder, ReconErrorSpec

sns.set(style='whitegrid', palette='muted', font_scale=1.2)
COLORS_PALETTE = ["#01BEFE", "#FFDD00", "#FF7D00", "#FF006D", "#ADFF02", "#8F00FF"]
sns.set_palette(sns.color_palette(COLORS_PALETTE))
rcParams['figure.figsize'] = 12, 8

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


In [ ]:
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = torch.device("cpu")


## Loading Data

In [ ]:
# Load the files into Pandas data frames
train_fname = "SubFeatures_baseline3hr_zoneA.csv"
test_fname = "SubFeatures_cyberphysical_0_1_zoneA.csv"
labels_fname = "CyPhyScenarioLabels_2Class_0_1_zoneA.csv"

train = pd.read_csv(train_fname)
test = pd.read_csv(test_fname)
labels = pd.read_csv(labels_fname)

# 1. Append a label column to train where every label is 0 (normal class)
train['target'] = 0  # directly name it 'target' to avoid confusion

# 2. Merge test with labels on 'timestamp' to add labels
test = test.merge(labels, on='timestamp', how='left')

# Check for missing labels after merge
missing_labels = test['labels'].isna().sum()
print(f"Number of missing labels in test after merge: {missing_labels}")

# Optionally fill missing labels with a default value (e.g., -1 or 1)
# For example, if missing means abnormal:
test['labels'].fillna(1, inplace=True)

# Rename 'labels' column in test to 'target'
test.rename(columns={'labels': 'target'}, inplace=True)

# Now concatenate train and test
df = pd.concat([train, test])

# Shuffle the combined dataframe
df = df.sample(frac=1.0).reset_index(drop=True)

# Drop unnecessary columns
df.drop('timestamp', axis=1, inplace=True)
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)

# Confirm the target column exists and no 'label' or 'labels' column remains
print("Columns after processing:", df.columns)
print("Dataframe Shape: ", df.shape)
print("Dataframe Head: ")
print(df.head())

CLASS_NORMAL = 0
class_names = ['Normal', 'Abnormal']

In [ ]:
# Check for NaN values
nan_mask = df.isna()


# Check if any NaN values exist in the DataFrame
if nan_mask.any().any():
    print("\nDataFrame contains NaN values.")
else:
    print("\nDataFrame does not contain any NaN values.")



In [ ]:
# Display rows with NaN values
rows_with_nan = df[nan_mask.any(axis=1)]
print("\nRows with NaN values:")
print(len(rows_with_nan))

In [ ]:
df.dropna(inplace=True)

In [ ]:
# Check for NaN values
nan_mask = df.isna()


# Check if any NaN values exist in the DataFrame
if nan_mask.any().any():
    print("\nDataFrame contains NaN values.")
else:
    print("\nDataFrame does not contain any NaN values.")


In [ ]:
cols_to_normalize = df.columns[:-1]
mask = df.iloc[:, -1] == 0
scaler = MinMaxScaler()
df.loc[mask,cols_to_normalize] = scaler.fit_transform(df.loc[mask,cols_to_normalize])

mask = df.iloc[:, -1] == 1
df.loc[mask,cols_to_normalize] = scaler.transform(df.loc[mask,cols_to_normalize])


print(df.head())
print(cols_to_normalize)

In [ ]:
#df[df.columns] = scaler.fit_transform(df[df.columns])
df.min()

In [ ]:
import pickle

# Saving the objects:
with open('zoneA_scaler_01_21_26.pkl', 'wb') as f:  # Python 3: open(..., 'wb')
    pickle.dump([scaler], f)

# # Getting back the objects:
# with open('objs.pkl') as f:  # Python 3: open(..., 'rb')
#     obj0, obj1, obj2 = pickle.load(f)


## EDA

In [ ]:
print("Examples for each class: ")
print(df.target.value_counts())
print(" ")

print("Plotting the above results - Visualization")
# ax = sns.countplot(df.target)
# ax.set_xticklabels(class_names);

df.target.value_counts().plot(kind='bar', figsize=(7, 6), rot=0);
plt.xlabel("Class", labelpad=14)
plt.ylabel("# Normal/Abnormal Packets", labelpad=14)
plt.title("Normal/Abnormal Instances", y=1.02);

In [ ]:
pd.options.display.max_columns = None

df1 = df[df.isna().any(axis=1)]
display(df1)
df1 = df1.dropna()
df1 = df1[df1.isna().any(axis=1)]
display(df1)

In [ ]:
# Averaged (smoothed out with one standard deviation on top and bottom of it) Time Series for each class
def plot_time_series_class(data, class_name, ax, n_steps=2):
    time_series_df = pd.DataFrame(data)

    smooth_path = time_series_df.rolling(n_steps).mean()
    path_deviation = 2 * time_series_df.rolling(n_steps).std()
    
    under_line = (smooth_path - path_deviation)[0]
    over_line = (smooth_path + path_deviation)[0]

    ax.plot(smooth_path, linewidth=4)
    ax.fill_between(
        path_deviation.index,
        under_line,
        over_line,
        alpha=.125
    )
    ax.set_title(class_name + ' Data', fontsize = 32)
    
    ax.xaxis.set_tick_params(labelsize=25)
    ax.yaxis.set_tick_params(labelsize=25)
    ax.set_xlabel('Data Sample', fontsize=32)
    ax.set_ylabel('Avg Time Series', fontsize=32)

classes = df.target.unique()

fig, axs = plt.subplots(
    nrows=len(classes) // 3+1,
    ncols=3,
    sharey=True,
    figsize=(14, 8)
)

for i,cls in enumerate(classes):
    ax = axs.flat[i]
    print(cls)
    
        
    data = df[df.target == cls] \
        .drop(labels='target', axis=1) \
            .mean(axis=1) \
                .to_numpy()
    plot_time_series_class(data, class_names[i], ax)

fig.delaxes(axs.flat[-1])
fig.tight_layout();

In [ ]:
# We will use the normal examples as training data for our model and record the reconstruction loss
normal_df = df[df.target == CLASS_NORMAL].drop(labels='target', axis=1)
print('Normal df Shape: ', normal_df.shape)
print(" ")

anomaly_df = df[df.target != CLASS_NORMAL].drop(labels='target', axis=1)
print('Abnormal df Shape: ', anomaly_df.shape)
print(" ")

# Split the normal examples into train, validation, and test sets
train_df, val_df = train_test_split(
    normal_df,
    test_size=0.7,
    random_state=RANDOM_SEED
)

val_df, test_df = train_test_split(
    val_df,
    test_size=0.7,
    random_state=RANDOM_SEED
)

#Create tensors from examples
def create_dataset(df):
    sequences = df.astype(np.float32).to_numpy().tolist()
    dataset = [torch.tensor(s).unsqueeze(1).float() for s in sequences]
    n_seq, seq_len, n_features = torch.stack(dataset).shape
    return dataset, seq_len, n_features

    
'''
Each Time Series will be converted to a 2D Tensor in the shape
sequence length x number of features [3x1]
'''

train_dataset, seq_len, n_features = create_dataset(train_df)
val_dataset, _, _ = create_dataset(val_df)
test_normal_dataset, _, _ = create_dataset(test_df)
test_anomaly_dataset, _, _ = create_dataset(anomaly_df)
 

In [ ]:
print(len(train_dataset))
print(np.shape(train_dataset[0]))
np.asarray(train_dataset[0]).ravel()[:50]

## LSTM-Based Autoencoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, seq_len, n_features, embedding_dim=16, dropout=0.1, l2_reg=0.01):
        super(Encoder, self).__init__()

        self.seq_len, self.n_features = seq_len, n_features
        self.embedding_dim, self.hidden_dim = embedding_dim, embedding_dim # Changed this so that hidden dimension aligns 
        self.l2_reg = l2_reg
        self.dropout = dropout

        self.rnn1 = nn.LSTM(
            input_size=n_features,
            hidden_size=self.hidden_dim,
            num_layers=1,
            batch_first=True
        )

        # Adding dropout to prevent overfitting
        self.dropout1 = nn.Dropout(p=dropout)

        self.rnn2 = nn.LSTM(
            input_size=self.hidden_dim,
            hidden_size=embedding_dim,
            num_layers=1,
            batch_first=True
        )

    def forward(self, x):
        # Dynamically determine batch size from input tensor
        batch_size = x.shape[0]

        # Reshape input tensor to match expected dimensions
        x = x.reshape((batch_size, self.seq_len, self.n_features))

        x, (_, _) = self.rnn1(x)
        x = self.dropout1(x)

        x, (hidden_n, _) = self.rnn2(x)

        # Adding L2 Regularization to the Loss function
        l2_loss = 0.0
        for param in self.parameters():
            l2_loss += torch.norm(param, p=2)

        return hidden_n.reshape((batch_size, self.embedding_dim)), self.l2_reg * l2_loss


In [ ]:
class Decoder(nn.Module):
    def __init__(self, seq_len, input_dim=16, n_features=1):
        super(Decoder, self).__init__()

        self.seq_len, self.input_dim = seq_len, input_dim
        self.hidden_dim, self.n_features = input_dim, n_features  # Changed this so that hidden dimension aligns 

        self.rnn1 = nn.LSTM(
            input_size=input_dim,
            hidden_size=input_dim,
            num_layers=1,
            batch_first=True
        )

        self.rnn2 = nn.LSTM(
            input_size=input_dim,
            hidden_size=self.hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.output_layer = nn.Linear(self.hidden_dim, n_features)

    def forward(self, x):
        # Dynamically determine batch size from input tensor
        batch_size = x.shape[0]

        # Repeat input tensor to match sequence length and features
        x = x.repeat(self.seq_len, self.n_features)
        x = x.reshape((batch_size, self.seq_len, self.input_dim))

        x, (hidden_n, cell_n) = self.rnn1(x)
        x, (hidden_n, cell_n) = self.rnn2(x)
        x = x.reshape((batch_size, self.seq_len, self.hidden_dim))

        return self.output_layer(x)


In [ ]:
class RecurrentAutoencoder(nn.Module):
    def __init__(self, seq_len, n_features, embedding_dim=16):
        super(RecurrentAutoencoder, self).__init__()

        self.encoder = Encoder(seq_len, n_features, embedding_dim).to(device)
        self.decoder = Decoder(seq_len, embedding_dim, n_features).to(device)
        self.code = None
        self.l2_loss = 0

    def forward(self, x):
        x, self.l2_loss = self.encoder(x)
        self.code = x
        x = self.decoder(x)
        return x


In [ ]:
def train_model(model, train_dataset, val_dataset, n_epochs, lr, weight_decay, optimizer):

  global BATCH_SIZE
    
  optimizer = optimizer(model.parameters(), lr=lr)#, weight_decay=weight_decay)
  #criterion = nn.L1Loss(reduction='sum').to(device)
  criterion = nn.MSELoss().to(device)
  


  history = dict(train=[], val=[])

  best_model_wts = copy.deepcopy(model.state_dict())
  best_model_wts = None
  best_loss = 10000.0
  clip_value = 5
  
  
  for epoch in range(1, n_epochs + 1):
    
    model = model.train()    

    train_losses = []
        
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    for batch_idx, batch in enumerate(train_loader):
      seq_true = batch
      optimizer.zero_grad()

      seq_true = seq_true.to(device)
      if seq_true.shape[0] == BATCH_SIZE:
          
          seq_pred = model(seq_true)
          loss = criterion(seq_pred, seq_true) + model.l2_loss

          
          loss.backward()
          if criterion(seq_pred, seq_true).isnan():
                print("*** None!!! ***")
                #for name, param in model.named_parameters():
                #  if param.requires_grad:
                #    print (name, param.data)

          torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)
          optimizer.step()



          train_losses.append(loss.item())

    val_losses = []
    model = model.eval()
    
    train_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

    with torch.no_grad():
      for batch_idx, batch in enumerate(train_loader):

        seq_true = batch
        if seq_true.shape[0] == BATCH_SIZE:

            seq_true = seq_true.to(device)
            seq_pred = model(seq_true)

            loss = criterion(seq_pred, seq_true)
            val_losses.append(loss.item())

    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)

    history['train'].append(train_loss)
    history['val'].append(val_loss)

    if val_loss < best_loss:
      best_loss = val_loss
      best_model_wts = copy.deepcopy(model.state_dict())

    print(f'Epoch {epoch}: train loss {train_loss} val loss {val_loss}')
    
  model.load_state_dict(best_model_wts)
  return model.eval(), history 

In [ ]:
global BATCH_SIZE

best_model = None
best_history = {'train':[], 'val':[]}
best_history_avg_window = 10000

best_bs = None
best_lr = None
best_wd = None
best_opt = None
best_epochs = None

torch.autograd.set_detect_anomaly(True)

#for epochs in [150]:
for epochs in [20]:  # Run less epochs with a smaller batch size - more hard to run - less noisy - maybe faster convergence?
    for BATCH_SIZE in [16]:  # Try a batch size of 1 with smaller noise 
        for learning_rate in [1e-2]:
            for weight_decay in [0.01]:
                for optimizer in [torch.optim.Adagrad]:#, torch.optim.Adadelta, torch.optim.Adagrad, torch.optim.Rprop, torch.optim.Adam]:

                    print('--------------------------------------------------------------')
                    print('-----------------------Training Information-------------------')
                    print(' ')
                    print('Optimizer: ', optimizer)
                    print('Weight Decay: ', weight_decay)
                    print('Learning Rate: ', learning_rate)
                    print('Batch Size: ', BATCH_SIZE)
                    print(' ')
                    print(' ')

                    print('Starting LSTM AE Model Training...')
                    print(seq_len)
                    model = RecurrentAutoencoder(seq_len, n_features, 55)
                    model = model.to(device)
                    

                    model, history = train_model(
                      model, 
                      train_dataset, 
                      val_dataset, 
                      n_epochs=epochs,
                      lr=learning_rate,
                      weight_decay=weight_decay,
                      optimizer=optimizer
                    )

                    '''
                    Saving the LSTM AE with the best hyperparameters configuration

                    Idea: Check the average validation loss of the last K epochs. 
                          We should choose the model with the lowest validation loss,
                          as this indicates the best performance on unseen data.
                    '''
                    window_size = 10
                    history_window = sum(history['val'][-window_size:])/window_size

                    if history_window < best_history_avg_window:

                        print('*****************')
                        print('New Best Model...')
                        print('Optimizer: ', optimizer)
                        print('Weight Decay: ', weight_decay)
                        print('Learning Rate: ', learning_rate)
                        print('Batch Size: ', BATCH_SIZE)
                        print('Epochs: ', epochs)
                        print('*****************')

                        best_model = model
                        best_history['train'] = history['train']
                        best_history['val'] = history['val']
                        best_history_avg_window = history_window

                        best_bs = BATCH_SIZE
                        best_lr = learning_rate
                        best_wd = weight_decay
                        best_opt = optimizer
                        best_epochs = epochs


                        # Contents of best_history data structure -- Uncomment for testing purposes
                        # print(best_history['train'])

                    print('--------------------------------------------------------------')
                    print('--------------------------------------------------------------')
                    print(' ')
                    print(' ') 

## Testing Autoencoders

In [ ]:
ax = plt.figure().gca()

ax.plot(best_history['train'], linewidth=4)
ax.plot(best_history['val'], linewidth=4)

# Set y-axis limits
#ax.set_ylim([0.040, 0.125])  # Adjust the limits as needed


plt.ylabel('Loss', fontsize=32)
plt.xlabel('Epoch', fontsize=32)
ax.xaxis.set_tick_params(labelsize=25)
ax.yaxis.set_tick_params(labelsize=25)

plt.legend(['training loss', 'validation loss'], fontsize=30)
plt.title('Best Model - Loss over training epochs', fontsize=32)
plt.show(); 

In [ ]:
import pickle
with open('zone_A_LSTM_AE_train.pkl', 'wb') as file:
    # Step 4: Use pickle.dump() to save the list
    pickle.dump(best_history['train'], file) 

In [ ]:
with open('zone_A_LSTM_AE_val.pkl', 'wb') as file:
    # Step 4: Use pickle.dump() to save the list
    pickle.dump(best_history['val'], file) 

In [ ]:
MODEL_PATH = 'zoneA_ae_01_21_26.pth'

torch.save(best_model, MODEL_PATH) 

In [ ]:
def predict(model, dataset):
    predictions, losses = [], []
    predictions_data_points = []
    real_data_points = []
    loss_data_points = []
    code_data_points = []
    predictions_flatten = []
    initial_features_flatten = []

    criterion = nn.MSELoss().to(device)
    with torch.no_grad():
        model = model.eval()

        # Iterate over each data point in the dataset
        for idx in range(len(dataset)):
            # Extract a single data point
            data_point = dataset[idx]

            # Convert to PyTorch tensor if necessary
            if isinstance(data_point, list):
                data_point = torch.tensor(data_point, dtype=torch.float32)
            elif isinstance(data_point, np.ndarray):
                data_point = torch.tensor(data_point, dtype=torch.float32)

            # Reshape the single data point to simulate a batch of size 1
            data_point = data_point.unsqueeze(0).to(device)  # Add batch dimension

            # Forward pass
            seq_pred = model(data_point)

            # Collect predictions and real data
            predictions_data_points.append(seq_pred[0])  # Remove batch dimension
            real_data_points.append(data_point[0])  # Remove batch dimension

            # Collect additional information
            code_data_points.append(model.code[0].cpu().numpy().flatten())  # Sending it to CPU
            predictions_flatten.append(seq_pred[0].cpu().numpy().flatten())  # Sending it to CPU
            initial_features_flatten.append(data_point[0].cpu().numpy().flatten())  # Sending it to CPU

            # Compute loss
            loss = criterion(seq_pred.cpu(), data_point.cpu())  # Sending to CPU
            loss_data_points.append(loss.item())

            # Flatten predictions for overall collection
            predictions.append(seq_pred.cpu().numpy().flatten())  # Sending to CPU
            losses.append(loss.item())

    return predictions, losses, predictions_data_points, real_data_points, loss_data_points, predictions_flatten, code_data_points, initial_features_flatten


In [ ]:
BATCH_SIZE = 1
predictions, losses, _, _, losses_data, _, _, _ = predict(best_model, train_dataset)

print("Number of Predictions:", len(predictions))

ax = sns.distplot(losses_data, bins=90, kde=True)
ax.set(xlabel='RE', ylabel='Normal Data Density')
sns.set(font_scale=3)


In [ ]:
THRESHOLD = np.percentile(losses, 99)
THRESHOLD

In [ ]:
# Normal Data Predictions [~100%]
predictions, pred_losses,  _, _, losses_data, _, _, _= predict(best_model, test_normal_dataset)
correct = sum(l <= THRESHOLD for l in losses_data)
print(f'Correct Normal predictions: {correct}/{len(losses_data)}') 

In [ ]:
# Abnormal Data Predictions 
anomaly_dataset = test_anomaly_dataset[:len(test_normal_dataset)]
predictions, pred_losses,  _, _, losses_data, _, _, _= predict(best_model, anomaly_dataset)
ax = sns.distplot(losses_data, bins=90, kde=True);

ax.set(xlabel='RE', ylabel='Abnormal Data Density')
sns.set(font_scale=3) 

In [ ]:
# (~100%)
correct = sum(l > THRESHOLD for l in losses_data)
print(f'Correct anomaly predictions: {correct}/{len(losses_data)}') 

## Verification Pipeline

In [ ]:
# Checking for certain dimensions
# print("num windows:", len(train_dataset))
# print("one window shape:", np.asarray(train_dataset[0]).shape)
# print("stacked shape:", np.stack([np.asarray(w) for w in train_dataset]).shape)

In [ ]:
# convert the trained torch model -> cert-rnn (captures the full stacks)
def cell(r):
    c = nn.LSTMCell(r.input_size, r.hidden_size).to(r.weight_ih_l0.dtype)
    with torch.no_grad():
        c.weight_ih.copy_(r.weight_ih_l0); c.weight_hh.copy_(r.weight_hh_l0)
        c.bias_ih.copy_(r.bias_ih_l0);     c.bias_hh.copy_(r.bias_hh_l0)
    return c

ae = LSTMAutoencoder.from_torch(
    [cell(model.encoder.rnn1), cell(model.encoder.rnn2)],   # full encoder stack
    [cell(model.decoder.rnn1), cell(model.decoder.rnn2)],   # full decoder stack
    model.decoder.output_layer)

print("captured: ", ae.encoder["L"], "enc layers,", ae.decoder["L"], "dec layers", " D=", ae.D, " H=", ae.H)

In [ ]:
X_val = np.stack([np.asarray(val_dataset[i], dtype=np.float64).reshape(-1,ae.D) for i in range(len(val_dataset))])
scores = ae.score(X_val)
tau = float(np.percentile(scores,95))
i = int(np.argmin(scores))
anchor = X_val[i]
print(f"tau={tau:.5f} anchor#{i} score={scores[i]:.5f} (< tau: {scores[i] < tau})")

In [ ]:
report = ae.preflight(anchor, tau=tau, torch_model=model)
print(report)
assert report.ok

In [ ]:
smoke = ae.smoke_test(anchor,tau,n_frames=16)
assert smoke["ok"]
print(f"one full-length reach ~{smoke['est_sec_per_reach_full']:.0f}s")
print(f"certify multi_frame ~{smoke['est_certify_multi_frame_s']/60:.1f} min")
print(f"certify single_frame ~{smoke['est_certify_single_frame_s'] / 3600:.1f} h (per-frame profile)")


In [ ]:
# Export from notebook to script 
torch.save(model.state_dict(), "lstm_ae_checkpoint.pt")
np.savez("verify_inputs.npz", anchor=anchor, tau=tau)
print("Exported: ")
print("   lstm_ae_checkpoint.pt keys:", len(model.state_dict()), "tensors")
print("   verify_inputs.npz     anchor", anchor.shape, " tau", round(tau,5))

In [ ]:
# To run inside the notebook (takes about 11 minutes)
#  res = ae.certify(anchor, ReconErrorSpec(tau), threat_model="multi_frame")
# print(res)
